# 서울시 대기질 추세 분석 — 2019~2026

## 목적

이 노트북은 **서울시 대기질의 장기·계절·시간대 추세를 시각적으로 파악**하기 위한 EDA 노트북이다.

분석 단위는 **측정소명 = 자치구 단위**로 본다. 이번 단계에서는 기상 데이터와 결합하지 않고 대기질만 독립적으로 분석한다.

### 분석 대상 오염물질

1. 미세먼지 1시간
2. 미세먼지 24시간
3. 초미세먼지
4. 오존
5. 이산화질소
6. 일산화탄소
7. 아황산가스

---

## 분석 파이프라인

```text
[2019~2026 서울 대기질 원본]
        │
        ├─ 2019~2025 : 연도별 CSV
        └─ 2026      : OpenAPI 수집 CSV
        ↓
[컬럼명 통일]
        ↓
[데이터 품질 점검]
        │
        ├─ 중복
        ├─ 결측
        ├─ 음수
        ├─ 0값
        └─ 측정소 수 / 시간 범위
        ↓
[시간 해상도별 집계]
        │
        ├─ HOURLY : 원 시계열
        ├─ DAILY  : 일평균
        └─ MONTHLY: 월평균
        ↓
[시각화 중심 EDA]
        │
        ├─ 측정소 × 오염물질 상세 Explorer
        │    ├─ 시간별 상세선
        │    ├─ 일별 추세선
        │    └─ 월별 장기 추세선
        ├─ 평균적인 24시간 패턴
        ├─ 평균적인 1~12월 계절성
        ├─ 연도별 월 패턴
        ├─ 25개 구 월별 Heatmap
        └─ 서울 평균 대비 지역 편차
        ↓
[추세 해석]
        │
        ├─ 장기 추세가 존재하는가?
        ├─ 계절성이 반복되는가?
        ├─ 시간대 패턴이 존재하는가?
        ├─ 지역 차이가 지속적인가?
        └─ 지역 순위가 시기에 따라 바뀌는가?
```

### 왜 모든 그래프를 한 번에 출력하지 않는가?

25개 측정소 × 7개 오염물질 × 3개 시간 해상도를 모두 출력하면 최소 525개 이상의 그래프가 생긴다.

따라서 이 노트북은 다음 원칙을 사용한다.

- **상세 분석:** 측정소와 오염물질을 선택해서 본다.
- **전체 비교:** Heatmap / 서울 평균 대비 편차 / 계절 프로파일을 이용한다.
- **시간별 raw line:** 선택 기간만 본다.
- **일별 line:** 수개월~수년의 흐름을 본다.
- **월별 line:** 2019~2026 장기 추세를 본다.

### 2026년 해석 주의

2026년은 완전연도가 아닐 수 있으므로 2019~2025 완전연도 비교와 2026 현재까지의 흐름을 구분해서 해석한다.


> **확장 버전:** 모든 측정소 × 모든 오염물질 조합의 상세 그래프를 자동 생성한다.

> **최종 전체 조합 버전:** 25개 실제 측정소명 × 모든 오염물질에 대해 10~14번 시각화를 자동 생성한다.

## 1. 라이브러리

In [ ]:
from pathlib import Path
import re
import unicodedata
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 180)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

## 2. 파일 경로

원본 대기질 CSV는 연도별로 한 곳에 통일한다.

```text
Toyproject/
├─ raw_air/
│  └─ yearly/
│     ├─ seoul_air_2019.csv
│     ├─ seoul_air_2020.csv
│     ├─ ...
│     ├─ seoul_air_2025.csv
│     └─ seoul_air_2026.csv
│
└─ air_quality_analysis/
   ├─ air_hourly_clean_2019_2026.csv
   ├─ air_daily_mean_2019_2026.csv
   ├─ air_monthly_mean_2019_2026.csv
   ├─ air_quality_summary_2019_2026.csv
   └─ air_year_station_summary_2019_2026.csv
```

- `raw_air/yearly`: 원본 데이터
- `air_quality_analysis`: 이 노트북이 생성하는 정제·집계 데이터


In [ ]:
PROJECT_DIR = Path(
    r"C:\Users\Luke\Desktop\DArtB\Toyproject"
)

RAW_DIR = PROJECT_DIR / "raw_air"
YEARLY_DIR = RAW_DIR / "yearly"

ANALYSIS_DIR = (
    PROJECT_DIR
    / "air_quality_analysis"
)

ANALYSIS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

YEARS = range(
    2019,
    2027
)

print("PROJECT_DIR :", PROJECT_DIR)
print("RAW_DIR     :", RAW_DIR)
print("YEARLY_DIR  :", YEARLY_DIR)
print("ANALYSIS_DIR:", ANALYSIS_DIR)

## 2-1. 한글 폰트 설정

Matplotlib 그래프에서 한글이 깨지지 않도록 운영체제별 기본 한글 폰트를 설정한다.

- Windows: `Malgun Gothic`
- macOS: `AppleGothic`
- Linux/Colab: `NanumGothic`

마이너스 기호 깨짐도 함께 방지한다.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform

system_name = platform.system()

if system_name == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"

elif system_name == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"

else:
    plt.rcParams["font.family"] = "NanumGothic"

plt.rcParams["axes.unicode_minus"] = False

print("한글 폰트 설정 완료:", plt.rcParams["font.family"])

## 3. 2019~2026 파일 자동 탐색 및 로드

In [ ]:
ENCODINGS = [
    "utf-8-sig",
    "utf-8",
    "cp949",
    "euc-kr"
]


def read_csv_safe(path):

    last_error = None

    for enc in ENCODINGS:

        try:

            return pd.read_csv(
                path,
                encoding=enc,
                low_memory=False
            )

        except Exception as e:

            last_error = e

    raise RuntimeError(
        f"CSV 읽기 실패: {path}\n{last_error}"
    )


def find_year_file(year):

    # 기본 파일명
    path = (
        YEARLY_DIR
        / f"seoul_air_{year}.csv"
    )

    if path.exists():

        return path


    # 혹시 파일명이 약간 다를 경우
    matches = sorted(
        p
        for p in YEARLY_DIR.glob("*.csv")
        if str(year) in p.name
        and "air" in p.name.lower()
    )

    if matches:

        return matches[0]


    return None


year_files = {}


for year in YEARS:

    path = find_year_file(
        year
    )

    if path is None:

        print(
            f"✗ {year}: 파일 없음"
        )

    else:

        year_files[
            year
        ] = path

        print(
            f"✓ {year}: {path}"
        )

## 4. 연도별 컬럼명 통일

서울시 과거 CSV와 2026 OpenAPI 파일의 컬럼명이 다를 수 있으므로 내부 표준 컬럼으로 통일한다.

| 표준 컬럼 | 의미 |
|---|---|
| `datetime` | 측정일시 |
| `station` | 측정소명 / 자치구 |
| `pm10_1h` | 미세먼지 1시간 |
| `pm10_24h` | 미세먼지 24시간 |
| `pm25` | 초미세먼지 |
| `o3` | 오존 |
| `no2` | 이산화질소 |
| `co` | 일산화탄소 |
| `so2` | 아황산가스 |


In [ ]:
COLUMN_ALIASES = {
    'datetime': ['datetime', 'MSRMT_DT', '측정일시', '측정일자'],
    'station': ['MSRSTN_NM', '측정소명', '측정소'],
    'pm10_1h': ['PM_HOUR', '미세먼지1시간(㎍/㎥)', '미세먼지 1시간(㎍/㎥)', '미세먼지1시간(ug/m3)', '미세먼지 1시간'],
    'pm10_24h': ['PM_DAY', '미세먼지24시간(㎍/㎥)', '미세먼지 24시간(㎍/㎥)', '미세먼지24시간(ug/m3)', '미세먼지 24시간'],
    'pm25': ['FPM', '초미세먼지(㎍/㎥)', '초미세먼지(ug/m3)', '초미세먼지'],
    'o3': ['OZON', '오존(ppm)', '오존'],
    'no2': ['NTDX', '이산화질소농도(ppm)', '이산화질소(ppm)', '이산화질소농도', '이산화질소'],
    'co': ['CBMX', '일산화탄소농도(ppm)', '일산화탄소(ppm)', '일산화탄소농도', '일산화탄소'],
    'so2': ['SPDX', '아황산가스농도(ppm)', '아황산가스(ppm)', '아황산가스농도', '아황산가스'],
}

def normalize_label(text):
    text = unicodedata.normalize('NFKC', str(text))
    text = text.lower().strip()
    text = re.sub(r'\s+', '', text)
    text = text.replace('μ', 'u').replace('㎍', 'ug')
    text = re.sub(r'[^0-9a-z가-힣]+', '', text)
    return text

def find_column(df, aliases):
    lookup = {normalize_label(col): col for col in df.columns}
    for alias in aliases:
        key = normalize_label(alias)
        if key in lookup:
            return lookup[key]
    return None

def standardize_one_year(df, year):
    rename_map = {}
    for standard_name, aliases in COLUMN_ALIASES.items():
        found = find_column(df, aliases)
        if found is not None:
            rename_map[found] = standard_name

    out = df.rename(columns=rename_map).copy()
    missing_required = [c for c in ['datetime', 'station'] if c not in out.columns]
    if missing_required:
        raise ValueError(
            f'{year}: 필수 컬럼 누락 {missing_required}\n현재 컬럼: {out.columns.tolist()}'
        )

    raw_dt = out['datetime'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
    parsed_numeric = pd.to_datetime(raw_dt, format='%Y%m%d%H%M', errors='coerce')
    parsed_general = pd.to_datetime(raw_dt, errors='coerce')
    out['datetime'] = parsed_numeric.fillna(parsed_general)

    pollutant_cols = ['pm10_1h', 'pm10_24h', 'pm25', 'o3', 'no2', 'co', 'so2']
    for col in pollutant_cols:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors='coerce')

    out['station'] = out['station'].astype(str).str.strip()
    out['source_year'] = year
    keep = ['datetime', 'station'] + [c for c in pollutant_cols if c in out.columns] + ['source_year']
    return out[keep]

## 5. 전체 데이터 통합

In [ ]:
frames = []
load_summary = []

for year, path in year_files.items():
    df_raw = read_csv_safe(path)
    try:
        df_std = standardize_one_year(df_raw, year)
        frames.append(df_std)
        load_summary.append({
            'year': year,
            'file': path.name,
            'rows': len(df_std),
            'start': df_std['datetime'].min(),
            'end': df_std['datetime'].max(),
            'stations': df_std['station'].nunique(),
            'columns': ', '.join(df_std.columns)
        })
        print(f"✓ {year}: {len(df_std):,} rows / {df_std['station'].nunique()} stations")
    except Exception as e:
        print(f'✗ {year}: {e}')

load_summary = pd.DataFrame(load_summary)
display(load_summary)

if not frames:
    raise RuntimeError('로드된 데이터가 없습니다. 경로와 컬럼명을 확인하세요.')

air = pd.concat(frames, ignore_index=True, sort=False)
print('\n통합 shape:', air.shape)
print('기간:', air['datetime'].min(), '~', air['datetime'].max())
print('측정소 수:', air['station'].nunique())

# 6. 데이터 품질 점검

추세 분석 전에 중복, 결측, 음수, 0값, 측정소 수를 확인한다.

### 0값 처리
0이 실제 측정값인지 결측 대체인지 확정 전에는 기본적으로 유지한다. 필요하면 `ZERO_AS_NAN = True`로 민감도 비교를 할 수 있다.


In [ ]:
POLLUTANTS = {
    'pm10_1h': '미세먼지 1시간 (㎍/㎥)',
    'pm10_24h': '미세먼지 24시간 (㎍/㎥)',
    'pm25': '초미세먼지 (㎍/㎥)',
    'o3': '오존 (ppm)',
    'no2': '이산화질소 (ppm)',
    'co': '일산화탄소 (ppm)',
    'so2': '아황산가스 (ppm)'
}
AVAILABLE_POLLUTANTS = [col for col in POLLUTANTS if col in air.columns]
print('사용 가능한 오염물질:')
for col in AVAILABLE_POLLUTANTS:
    print('-', col, ':', POLLUTANTS[col])

In [ ]:
quality_rows = []
for col in AVAILABLE_POLLUTANTS:
    s = air[col]
    quality_rows.append({
        'variable': col,
        'label': POLLUTANTS[col],
        'count': int(s.notna().sum()),
        'missing_count': int(s.isna().sum()),
        'missing_rate': float(s.isna().mean()),
        'zero_count': int((s == 0).sum()),
        'zero_rate': float((s == 0).mean()),
        'negative_count': int((s < 0).sum()),
        'mean': s.mean(),
        'median': s.median(),
        'p99': s.quantile(.99),
        'max': s.max()
    })
quality_df = pd.DataFrame(quality_rows)
display(quality_df)

print('datetime × station 중복:', air.duplicated(subset=['datetime', 'station']).sum())

year_station_summary = (
    air.assign(year=air['datetime'].dt.year)
    .groupby('year')
    .agg(
        rows=('station', 'size'),
        station_count=('station', 'nunique'),
        start=('datetime', 'min'),
        end=('datetime', 'max')
    )
    .reset_index()
)
display(year_station_summary)

## 6.1 분석용 정제

In [ ]:
ZERO_AS_NAN = False

air_clean = air.copy()
air_clean = air_clean[air_clean['datetime'].notna() & air_clean['station'].notna()].copy()
air_clean = air_clean.drop_duplicates()

for col in AVAILABLE_POLLUTANTS:
    air_clean.loc[air_clean[col] < 0, col] = np.nan
    if ZERO_AS_NAN:
        air_clean.loc[air_clean[col] == 0, col] = np.nan

agg_dict = {col: 'mean' for col in AVAILABLE_POLLUTANTS}
air_clean = (
    air_clean
    .groupby(['datetime', 'station'], as_index=False)
    .agg(agg_dict)
    .sort_values(['datetime', 'station'])
    .reset_index(drop=True)
)

air_clean['year'] = air_clean['datetime'].dt.year
air_clean['month'] = air_clean['datetime'].dt.month
air_clean['hour'] = air_clean['datetime'].dt.hour
air_clean['date'] = air_clean['datetime'].dt.floor('D')
air_clean['year_month'] = air_clean['datetime'].dt.to_period('M').astype(str)

print('정제 후 shape:', air_clean.shape)
print('측정소 수:', air_clean['station'].nunique())
print('기간:', air_clean['datetime'].min(), '~', air_clean['datetime'].max())

## 7. 측정소 목록 확인

In [ ]:
stations = sorted(
    air_clean["station"]
    .dropna()
    .unique()
)

print("측정소 수:", len(stations))

station_code_map = pd.DataFrame({
    "code": range(len(stations)),
    "station": stations
})

display(station_code_map)

# 8. 시간 해상도별 집계

- **HOURLY**: 원시계열, 특정 기간 상세 관찰
- **DAILY**: 일평균, 중기 흐름
- **MONTHLY**: 월평균, 2019~2026 장기 추세와 계절성


In [ ]:
hourly = air_clean.copy()

daily = (
    air_clean
    .groupby(['station', 'date'], as_index=False)[AVAILABLE_POLLUTANTS]
    .mean()
)
daily['year'] = daily['date'].dt.year
daily['month'] = daily['date'].dt.month

monthly = (
    air_clean
    .groupby(
        ['station', pd.Grouper(key='datetime', freq='MS')],
        as_index=False
    )[AVAILABLE_POLLUTANTS]
    .mean()
)
monthly['year'] = monthly['datetime'].dt.year
monthly['month'] = monthly['datetime'].dt.month
monthly['year_month'] = monthly['datetime'].dt.to_period('M').astype(str)

print('HOURLY :', hourly.shape)
print('DAILY  :', daily.shape)
print('MONTHLY:', monthly.shape)

# 9. 시각화 함수

In [ ]:
def validate_station_pollutant(station, pollutant):
    if station not in stations:
        raise ValueError(f'없는 측정소: {station}\n사용 가능: {stations}')
    if pollutant not in AVAILABLE_POLLUTANTS:
        raise ValueError(f'없는 오염물질: {pollutant}\n사용 가능: {AVAILABLE_POLLUTANTS}')


def plot_hourly(station, pollutant, start=None, end=None, rolling_hours=None):
    validate_station_pollutant(station, pollutant)
    temp = hourly[hourly['station'] == station].copy()
    if start is not None:
        temp = temp[temp['datetime'] >= pd.Timestamp(start)]
    if end is not None:
        temp = temp[temp['datetime'] <= pd.Timestamp(end)]
    temp = temp.sort_values('datetime')

    plt.figure(figsize=(14, 5))
    plt.plot(temp['datetime'], temp[pollutant], linewidth=0.8, label='Hourly')

    if rolling_hours is not None:
        roll = (
            temp.set_index('datetime')[pollutant]
            .rolling(rolling_hours, min_periods=max(1, rolling_hours // 4))
            .mean()
        )
        plt.plot(roll.index, roll.values, linewidth=1.5, label=f'{rolling_hours}h rolling mean')
        plt.legend()

    plt.title(f'{station} | {POLLUTANTS[pollutant]} | 시간별')
    plt.xlabel('Datetime')
    plt.ylabel(POLLUTANTS[pollutant])
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()


def plot_daily(station, pollutant, start=None, end=None, rolling_days=7):
    validate_station_pollutant(station, pollutant)
    temp = daily[daily['station'] == station].copy()
    if start is not None:
        temp = temp[temp['date'] >= pd.Timestamp(start)]
    if end is not None:
        temp = temp[temp['date'] <= pd.Timestamp(end)]
    temp = temp.sort_values('date')

    plt.figure(figsize=(14, 5))
    plt.plot(temp['date'], temp[pollutant], linewidth=0.7, alpha=0.55, label='Daily mean')

    if rolling_days is not None:
        roll = (
            temp.set_index('date')[pollutant]
            .rolling(rolling_days, min_periods=max(1, rolling_days // 2))
            .mean()
        )
        plt.plot(roll.index, roll.values, linewidth=1.5, label=f'{rolling_days}d rolling mean')
        plt.legend()

    plt.title(f'{station} | {POLLUTANTS[pollutant]} | 일별')
    plt.xlabel('Date')
    plt.ylabel(POLLUTANTS[pollutant])
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()


def plot_monthly(station, pollutant):
    validate_station_pollutant(station, pollutant)
    temp = monthly[monthly['station'] == station].sort_values('datetime')

    plt.figure(figsize=(14, 5))
    plt.plot(temp['datetime'], temp[pollutant], marker='o', markersize=2.5, linewidth=1.1)
    plt.title(f'{station} | {POLLUTANTS[pollutant]} | 월별 장기 추세')
    plt.xlabel('Month')
    plt.ylabel(POLLUTANTS[pollutant])
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()


def plot_all_scales(station, pollutant, hourly_start=None, hourly_end=None, daily_start=None, daily_end=None):
    plot_hourly(station, pollutant, start=hourly_start, end=hourly_end, rolling_hours=24)
    plot_daily(station, pollutant, start=daily_start, end=daily_end, rolling_days=7)
    plot_monthly(station, pollutant)

# 10. 전체 측정소 × 전체 오염물질 상세 Explorer

측정소나 오염물질 파라미터를 직접 바꾸지 않는다.

**25개 실제 측정소명 × 사용 가능한 모든 오염물질** 조합을 자동으로 실행한다.

각 조합마다:

1. 시간별 상세 추세 — 최근 30일 + 24시간 이동평균
2. 일별 장기 추세 — 2019~2026 + 7일 이동평균
3. 월별 장기 추세 — 2019~2026

를 출력한다.


In [ ]:
DETAIL_LAST_DT = hourly["datetime"].max()

DETAIL_HOURLY_END = DETAIL_LAST_DT
DETAIL_HOURLY_START = (
    DETAIL_LAST_DT
    - pd.Timedelta(days=30)
)

DETAIL_DAILY_START = (
    air_clean["datetime"]
    .min()
    .floor("D")
)

DETAIL_DAILY_END = (
    air_clean["datetime"]
    .max()
    .floor("D")
)

print("시간별 상세 구간:")
print(
    DETAIL_HOURLY_START,
    "~",
    DETAIL_HOURLY_END
)

print("\n일별/월별 전체 분석 구간:")
print(
    DETAIL_DAILY_START,
    "~",
    DETAIL_DAILY_END
)

print("\n전체 조합 수:")
print(
    len(stations),
    "×",
    len(AVAILABLE_POLLUTANTS),
    "=",
    len(stations) * len(AVAILABLE_POLLUTANTS)
)

## 10.1 강남구

**강남구 × 모든 오염물질**

각 오염물질에 대해 시간별 → 일별 → 월별 추세를 연속으로 출력한다.


In [ ]:
station_name = "강남구"

if station_name not in stations:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

else:

    print(
        "\n" + "=" * 100
    )

    print(
        "측정소:",
        station_name
    )

    print(
        "=" * 100
    )

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            "\n" + "-" * 90
        )

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        print(
            "-" * 90
        )

        plot_hourly(
            station_name,
            pollutant,
            start=DETAIL_HOURLY_START,
            end=DETAIL_HOURLY_END,
            rolling_hours=24
        )

        plot_daily(
            station_name,
            pollutant,
            start=DETAIL_DAILY_START,
            end=DETAIL_DAILY_END,
            rolling_days=7
        )

        plot_monthly(
            station_name,
            pollutant
        )

## 10.2 강동구

**강동구 × 모든 오염물질**

각 오염물질에 대해 시간별 → 일별 → 월별 추세를 연속으로 출력한다.


In [ ]:
station_name = "강동구"

if station_name not in stations:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

else:

    print(
        "\n" + "=" * 100
    )

    print(
        "측정소:",
        station_name
    )

    print(
        "=" * 100
    )

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            "\n" + "-" * 90
        )

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        print(
            "-" * 90
        )

        plot_hourly(
            station_name,
            pollutant,
            start=DETAIL_HOURLY_START,
            end=DETAIL_HOURLY_END,
            rolling_hours=24
        )

        plot_daily(
            station_name,
            pollutant,
            start=DETAIL_DAILY_START,
            end=DETAIL_DAILY_END,
            rolling_days=7
        )

        plot_monthly(
            station_name,
            pollutant
        )

## 10.3 강북구

**강북구 × 모든 오염물질**

각 오염물질에 대해 시간별 → 일별 → 월별 추세를 연속으로 출력한다.


In [ ]:
station_name = "강북구"

if station_name not in stations:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

else:

    print(
        "\n" + "=" * 100
    )

    print(
        "측정소:",
        station_name
    )

    print(
        "=" * 100
    )

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            "\n" + "-" * 90
        )

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        print(
            "-" * 90
        )

        plot_hourly(
            station_name,
            pollutant,
            start=DETAIL_HOURLY_START,
            end=DETAIL_HOURLY_END,
            rolling_hours=24
        )

        plot_daily(
            station_name,
            pollutant,
            start=DETAIL_DAILY_START,
            end=DETAIL_DAILY_END,
            rolling_days=7
        )

        plot_monthly(
            station_name,
            pollutant
        )

## 10.4 강서구

**강서구 × 모든 오염물질**

각 오염물질에 대해 시간별 → 일별 → 월별 추세를 연속으로 출력한다.


In [ ]:
station_name = "강서구"

if station_name not in stations:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

else:

    print(
        "\n" + "=" * 100
    )

    print(
        "측정소:",
        station_name
    )

    print(
        "=" * 100
    )

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            "\n" + "-" * 90
        )

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        print(
            "-" * 90
        )

        plot_hourly(
            station_name,
            pollutant,
            start=DETAIL_HOURLY_START,
            end=DETAIL_HOURLY_END,
            rolling_hours=24
        )

        plot_daily(
            station_name,
            pollutant,
            start=DETAIL_DAILY_START,
            end=DETAIL_DAILY_END,
            rolling_days=7
        )

        plot_monthly(
            station_name,
            pollutant
        )

## 10.5 관악구

**관악구 × 모든 오염물질**

각 오염물질에 대해 시간별 → 일별 → 월별 추세를 연속으로 출력한다.


In [ ]:
station_name = "관악구"

if station_name not in stations:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

else:

    print(
        "\n" + "=" * 100
    )

    print(
        "측정소:",
        station_name
    )

    print(
        "=" * 100
    )

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            "\n" + "-" * 90
        )

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        print(
            "-" * 90
        )

        plot_hourly(
            station_name,
            pollutant,
            start=DETAIL_HOURLY_START,
            end=DETAIL_HOURLY_END,
            rolling_hours=24
        )

        plot_daily(
            station_name,
            pollutant,
            start=DETAIL_DAILY_START,
            end=DETAIL_DAILY_END,
            rolling_days=7
        )

        plot_monthly(
            station_name,
            pollutant
        )

## 10.6 광진구

**광진구 × 모든 오염물질**

각 오염물질에 대해 시간별 → 일별 → 월별 추세를 연속으로 출력한다.


In [ ]:
station_name = "광진구"

if station_name not in stations:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

else:

    print(
        "\n" + "=" * 100
    )

    print(
        "측정소:",
        station_name
    )

    print(
        "=" * 100
    )

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            "\n" + "-" * 90
        )

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        print(
            "-" * 90
        )

        plot_hourly(
            station_name,
            pollutant,
            start=DETAIL_HOURLY_START,
            end=DETAIL_HOURLY_END,
            rolling_hours=24
        )

        plot_daily(
            station_name,
            pollutant,
            start=DETAIL_DAILY_START,
            end=DETAIL_DAILY_END,
            rolling_days=7
        )

        plot_monthly(
            station_name,
            pollutant
        )

## 10.7 구로구

**구로구 × 모든 오염물질**

각 오염물질에 대해 시간별 → 일별 → 월별 추세를 연속으로 출력한다.


In [ ]:
station_name = "구로구"

if station_name not in stations:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

else:

    print(
        "\n" + "=" * 100
    )

    print(
        "측정소:",
        station_name
    )

    print(
        "=" * 100
    )

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            "\n" + "-" * 90
        )

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        print(
            "-" * 90
        )

        plot_hourly(
            station_name,
            pollutant,
            start=DETAIL_HOURLY_START,
            end=DETAIL_HOURLY_END,
            rolling_hours=24
        )

        plot_daily(
            station_name,
            pollutant,
            start=DETAIL_DAILY_START,
            end=DETAIL_DAILY_END,
            rolling_days=7
        )

        plot_monthly(
            station_name,
            pollutant
        )

## 10.8 금천구

**금천구 × 모든 오염물질**

각 오염물질에 대해 시간별 → 일별 → 월별 추세를 연속으로 출력한다.


In [ ]:
station_name = "금천구"

if station_name not in stations:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

else:

    print(
        "\n" + "=" * 100
    )

    print(
        "측정소:",
        station_name
    )

    print(
        "=" * 100
    )

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            "\n" + "-" * 90
        )

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        print(
            "-" * 90
        )

        plot_hourly(
            station_name,
            pollutant,
            start=DETAIL_HOURLY_START,
            end=DETAIL_HOURLY_END,
            rolling_hours=24
        )

        plot_daily(
            station_name,
            pollutant,
            start=DETAIL_DAILY_START,
            end=DETAIL_DAILY_END,
            rolling_days=7
        )

        plot_monthly(
            station_name,
            pollutant
        )

## 10.9 노원구

**노원구 × 모든 오염물질**

각 오염물질에 대해 시간별 → 일별 → 월별 추세를 연속으로 출력한다.


In [ ]:
station_name = "노원구"

if station_name not in stations:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

else:

    print(
        "\n" + "=" * 100
    )

    print(
        "측정소:",
        station_name
    )

    print(
        "=" * 100
    )

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            "\n" + "-" * 90
        )

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        print(
            "-" * 90
        )

        plot_hourly(
            station_name,
            pollutant,
            start=DETAIL_HOURLY_START,
            end=DETAIL_HOURLY_END,
            rolling_hours=24
        )

        plot_daily(
            station_name,
            pollutant,
            start=DETAIL_DAILY_START,
            end=DETAIL_DAILY_END,
            rolling_days=7
        )

        plot_monthly(
            station_name,
            pollutant
        )

## 10.10 도봉구

**도봉구 × 모든 오염물질**

각 오염물질에 대해 시간별 → 일별 → 월별 추세를 연속으로 출력한다.


In [ ]:
station_name = "도봉구"

if station_name not in stations:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

else:

    print(
        "\n" + "=" * 100
    )

    print(
        "측정소:",
        station_name
    )

    print(
        "=" * 100
    )

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            "\n" + "-" * 90
        )

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        print(
            "-" * 90
        )

        plot_hourly(
            station_name,
            pollutant,
            start=DETAIL_HOURLY_START,
            end=DETAIL_HOURLY_END,
            rolling_hours=24
        )

        plot_daily(
            station_name,
            pollutant,
            start=DETAIL_DAILY_START,
            end=DETAIL_DAILY_END,
            rolling_days=7
        )

        plot_monthly(
            station_name,
            pollutant
        )

## 10.11 동대문구

**동대문구 × 모든 오염물질**

각 오염물질에 대해 시간별 → 일별 → 월별 추세를 연속으로 출력한다.


In [ ]:
station_name = "동대문구"

if station_name not in stations:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

else:

    print(
        "\n" + "=" * 100
    )

    print(
        "측정소:",
        station_name
    )

    print(
        "=" * 100
    )

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            "\n" + "-" * 90
        )

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        print(
            "-" * 90
        )

        plot_hourly(
            station_name,
            pollutant,
            start=DETAIL_HOURLY_START,
            end=DETAIL_HOURLY_END,
            rolling_hours=24
        )

        plot_daily(
            station_name,
            pollutant,
            start=DETAIL_DAILY_START,
            end=DETAIL_DAILY_END,
            rolling_days=7
        )

        plot_monthly(
            station_name,
            pollutant
        )

## 10.12 동작구

**동작구 × 모든 오염물질**

각 오염물질에 대해 시간별 → 일별 → 월별 추세를 연속으로 출력한다.


In [ ]:
station_name = "동작구"

if station_name not in stations:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

else:

    print(
        "\n" + "=" * 100
    )

    print(
        "측정소:",
        station_name
    )

    print(
        "=" * 100
    )

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            "\n" + "-" * 90
        )

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        print(
            "-" * 90
        )

        plot_hourly(
            station_name,
            pollutant,
            start=DETAIL_HOURLY_START,
            end=DETAIL_HOURLY_END,
            rolling_hours=24
        )

        plot_daily(
            station_name,
            pollutant,
            start=DETAIL_DAILY_START,
            end=DETAIL_DAILY_END,
            rolling_days=7
        )

        plot_monthly(
            station_name,
            pollutant
        )

## 10.13 마포구

**마포구 × 모든 오염물질**

각 오염물질에 대해 시간별 → 일별 → 월별 추세를 연속으로 출력한다.


In [ ]:
station_name = "마포구"

if station_name not in stations:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

else:

    print(
        "\n" + "=" * 100
    )

    print(
        "측정소:",
        station_name
    )

    print(
        "=" * 100
    )

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            "\n" + "-" * 90
        )

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        print(
            "-" * 90
        )

        plot_hourly(
            station_name,
            pollutant,
            start=DETAIL_HOURLY_START,
            end=DETAIL_HOURLY_END,
            rolling_hours=24
        )

        plot_daily(
            station_name,
            pollutant,
            start=DETAIL_DAILY_START,
            end=DETAIL_DAILY_END,
            rolling_days=7
        )

        plot_monthly(
            station_name,
            pollutant
        )

## 10.14 서대문구

**서대문구 × 모든 오염물질**

각 오염물질에 대해 시간별 → 일별 → 월별 추세를 연속으로 출력한다.


In [ ]:
station_name = "서대문구"

if station_name not in stations:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

else:

    print(
        "\n" + "=" * 100
    )

    print(
        "측정소:",
        station_name
    )

    print(
        "=" * 100
    )

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            "\n" + "-" * 90
        )

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        print(
            "-" * 90
        )

        plot_hourly(
            station_name,
            pollutant,
            start=DETAIL_HOURLY_START,
            end=DETAIL_HOURLY_END,
            rolling_hours=24
        )

        plot_daily(
            station_name,
            pollutant,
            start=DETAIL_DAILY_START,
            end=DETAIL_DAILY_END,
            rolling_days=7
        )

        plot_monthly(
            station_name,
            pollutant
        )

## 10.15 서초구

**서초구 × 모든 오염물질**

각 오염물질에 대해 시간별 → 일별 → 월별 추세를 연속으로 출력한다.


In [ ]:
station_name = "서초구"

if station_name not in stations:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

else:

    print(
        "\n" + "=" * 100
    )

    print(
        "측정소:",
        station_name
    )

    print(
        "=" * 100
    )

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            "\n" + "-" * 90
        )

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        print(
            "-" * 90
        )

        plot_hourly(
            station_name,
            pollutant,
            start=DETAIL_HOURLY_START,
            end=DETAIL_HOURLY_END,
            rolling_hours=24
        )

        plot_daily(
            station_name,
            pollutant,
            start=DETAIL_DAILY_START,
            end=DETAIL_DAILY_END,
            rolling_days=7
        )

        plot_monthly(
            station_name,
            pollutant
        )

## 10.16 성동구

**성동구 × 모든 오염물질**

각 오염물질에 대해 시간별 → 일별 → 월별 추세를 연속으로 출력한다.


In [ ]:
station_name = "성동구"

if station_name not in stations:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

else:

    print(
        "\n" + "=" * 100
    )

    print(
        "측정소:",
        station_name
    )

    print(
        "=" * 100
    )

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            "\n" + "-" * 90
        )

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        print(
            "-" * 90
        )

        plot_hourly(
            station_name,
            pollutant,
            start=DETAIL_HOURLY_START,
            end=DETAIL_HOURLY_END,
            rolling_hours=24
        )

        plot_daily(
            station_name,
            pollutant,
            start=DETAIL_DAILY_START,
            end=DETAIL_DAILY_END,
            rolling_days=7
        )

        plot_monthly(
            station_name,
            pollutant
        )

## 10.17 성북구

**성북구 × 모든 오염물질**

각 오염물질에 대해 시간별 → 일별 → 월별 추세를 연속으로 출력한다.


In [ ]:
station_name = "성북구"

if station_name not in stations:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

else:

    print(
        "\n" + "=" * 100
    )

    print(
        "측정소:",
        station_name
    )

    print(
        "=" * 100
    )

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            "\n" + "-" * 90
        )

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        print(
            "-" * 90
        )

        plot_hourly(
            station_name,
            pollutant,
            start=DETAIL_HOURLY_START,
            end=DETAIL_HOURLY_END,
            rolling_hours=24
        )

        plot_daily(
            station_name,
            pollutant,
            start=DETAIL_DAILY_START,
            end=DETAIL_DAILY_END,
            rolling_days=7
        )

        plot_monthly(
            station_name,
            pollutant
        )

## 10.18 송파구

**송파구 × 모든 오염물질**

각 오염물질에 대해 시간별 → 일별 → 월별 추세를 연속으로 출력한다.


In [ ]:
station_name = "송파구"

if station_name not in stations:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

else:

    print(
        "\n" + "=" * 100
    )

    print(
        "측정소:",
        station_name
    )

    print(
        "=" * 100
    )

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            "\n" + "-" * 90
        )

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        print(
            "-" * 90
        )

        plot_hourly(
            station_name,
            pollutant,
            start=DETAIL_HOURLY_START,
            end=DETAIL_HOURLY_END,
            rolling_hours=24
        )

        plot_daily(
            station_name,
            pollutant,
            start=DETAIL_DAILY_START,
            end=DETAIL_DAILY_END,
            rolling_days=7
        )

        plot_monthly(
            station_name,
            pollutant
        )

## 10.19 양천구

**양천구 × 모든 오염물질**

각 오염물질에 대해 시간별 → 일별 → 월별 추세를 연속으로 출력한다.


In [ ]:
station_name = "양천구"

if station_name not in stations:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

else:

    print(
        "\n" + "=" * 100
    )

    print(
        "측정소:",
        station_name
    )

    print(
        "=" * 100
    )

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            "\n" + "-" * 90
        )

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        print(
            "-" * 90
        )

        plot_hourly(
            station_name,
            pollutant,
            start=DETAIL_HOURLY_START,
            end=DETAIL_HOURLY_END,
            rolling_hours=24
        )

        plot_daily(
            station_name,
            pollutant,
            start=DETAIL_DAILY_START,
            end=DETAIL_DAILY_END,
            rolling_days=7
        )

        plot_monthly(
            station_name,
            pollutant
        )

## 10.20 영등포구

**영등포구 × 모든 오염물질**

각 오염물질에 대해 시간별 → 일별 → 월별 추세를 연속으로 출력한다.


In [ ]:
station_name = "영등포구"

if station_name not in stations:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

else:

    print(
        "\n" + "=" * 100
    )

    print(
        "측정소:",
        station_name
    )

    print(
        "=" * 100
    )

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            "\n" + "-" * 90
        )

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        print(
            "-" * 90
        )

        plot_hourly(
            station_name,
            pollutant,
            start=DETAIL_HOURLY_START,
            end=DETAIL_HOURLY_END,
            rolling_hours=24
        )

        plot_daily(
            station_name,
            pollutant,
            start=DETAIL_DAILY_START,
            end=DETAIL_DAILY_END,
            rolling_days=7
        )

        plot_monthly(
            station_name,
            pollutant
        )

## 10.21 용산구

**용산구 × 모든 오염물질**

각 오염물질에 대해 시간별 → 일별 → 월별 추세를 연속으로 출력한다.


In [ ]:
station_name = "용산구"

if station_name not in stations:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

else:

    print(
        "\n" + "=" * 100
    )

    print(
        "측정소:",
        station_name
    )

    print(
        "=" * 100
    )

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            "\n" + "-" * 90
        )

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        print(
            "-" * 90
        )

        plot_hourly(
            station_name,
            pollutant,
            start=DETAIL_HOURLY_START,
            end=DETAIL_HOURLY_END,
            rolling_hours=24
        )

        plot_daily(
            station_name,
            pollutant,
            start=DETAIL_DAILY_START,
            end=DETAIL_DAILY_END,
            rolling_days=7
        )

        plot_monthly(
            station_name,
            pollutant
        )

## 10.22 은평구

**은평구 × 모든 오염물질**

각 오염물질에 대해 시간별 → 일별 → 월별 추세를 연속으로 출력한다.


In [ ]:
station_name = "은평구"

if station_name not in stations:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

else:

    print(
        "\n" + "=" * 100
    )

    print(
        "측정소:",
        station_name
    )

    print(
        "=" * 100
    )

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            "\n" + "-" * 90
        )

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        print(
            "-" * 90
        )

        plot_hourly(
            station_name,
            pollutant,
            start=DETAIL_HOURLY_START,
            end=DETAIL_HOURLY_END,
            rolling_hours=24
        )

        plot_daily(
            station_name,
            pollutant,
            start=DETAIL_DAILY_START,
            end=DETAIL_DAILY_END,
            rolling_days=7
        )

        plot_monthly(
            station_name,
            pollutant
        )

## 10.23 종로구

**종로구 × 모든 오염물질**

각 오염물질에 대해 시간별 → 일별 → 월별 추세를 연속으로 출력한다.


In [ ]:
station_name = "종로구"

if station_name not in stations:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

else:

    print(
        "\n" + "=" * 100
    )

    print(
        "측정소:",
        station_name
    )

    print(
        "=" * 100
    )

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            "\n" + "-" * 90
        )

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        print(
            "-" * 90
        )

        plot_hourly(
            station_name,
            pollutant,
            start=DETAIL_HOURLY_START,
            end=DETAIL_HOURLY_END,
            rolling_hours=24
        )

        plot_daily(
            station_name,
            pollutant,
            start=DETAIL_DAILY_START,
            end=DETAIL_DAILY_END,
            rolling_days=7
        )

        plot_monthly(
            station_name,
            pollutant
        )

## 10.24 중구

**중구 × 모든 오염물질**

각 오염물질에 대해 시간별 → 일별 → 월별 추세를 연속으로 출력한다.


In [ ]:
station_name = "중구"

if station_name not in stations:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

else:

    print(
        "\n" + "=" * 100
    )

    print(
        "측정소:",
        station_name
    )

    print(
        "=" * 100
    )

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            "\n" + "-" * 90
        )

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        print(
            "-" * 90
        )

        plot_hourly(
            station_name,
            pollutant,
            start=DETAIL_HOURLY_START,
            end=DETAIL_HOURLY_END,
            rolling_hours=24
        )

        plot_daily(
            station_name,
            pollutant,
            start=DETAIL_DAILY_START,
            end=DETAIL_DAILY_END,
            rolling_days=7
        )

        plot_monthly(
            station_name,
            pollutant
        )

## 10.25 중랑구

**중랑구 × 모든 오염물질**

각 오염물질에 대해 시간별 → 일별 → 월별 추세를 연속으로 출력한다.


In [ ]:
station_name = "중랑구"

if station_name not in stations:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

else:

    print(
        "\n" + "=" * 100
    )

    print(
        "측정소:",
        station_name
    )

    print(
        "=" * 100
    )

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            "\n" + "-" * 90
        )

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        print(
            "-" * 90
        )

        plot_hourly(
            station_name,
            pollutant,
            start=DETAIL_HOURLY_START,
            end=DETAIL_HOURLY_END,
            rolling_hours=24
        )

        plot_daily(
            station_name,
            pollutant,
            start=DETAIL_DAILY_START,
            end=DETAIL_DAILY_END,
            rolling_days=7
        )

        plot_monthly(
            station_name,
            pollutant
        )

# 11. 전체 측정소 × 전체 오염물질 평균 24시간 패턴

각 측정소·오염물질별로 **0~23시 평균 패턴**을 모두 생성한다.

이 그래프는 반복적인 시간대 특성을 보기 위한 핵심 시각화다.


In [ ]:
def plot_diurnal_profile(
    station,
    pollutant,
    year=None,
    month=None
):

    validate_station_pollutant(
        station,
        pollutant
    )

    temp = air_clean[
        air_clean["station"]
        == station
    ].copy()

    if year is not None:

        temp = temp[
            temp["year"]
            == year
        ]

    if month is not None:

        temp = temp[
            temp["month"]
            == month
        ]

    profile = (
        temp
        .groupby("hour")[pollutant]
        .mean()
        .reindex(range(24))
    )

    plt.figure(
        figsize=(11, 5)
    )

    plt.plot(
        profile.index,
        profile.values,
        marker="o"
    )

    extra = []

    if year is not None:
        extra.append(
            str(year)
        )

    if month is not None:
        extra.append(
            f"{month}월"
        )

    suffix = (
        " | "
        + " ".join(extra)
        if extra
        else ""
    )

    plt.title(
        f"{station} | "
        f"{POLLUTANTS[pollutant]} | "
        f"평균 24시간 패턴"
        f"{suffix}"
    )

    plt.xlabel("Hour")

    plt.ylabel(
        POLLUTANTS[pollutant]
    )

    plt.xticks(
        range(24)
    )

    plt.grid(
        alpha=0.25
    )

    plt.tight_layout()
    plt.show()

## 11.1 강남구

**강남구 × 모든 오염물질의 평균 24시간 패턴**


In [ ]:
station_name = "강남구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_diurnal_profile(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 11.2 강동구

**강동구 × 모든 오염물질의 평균 24시간 패턴**


In [ ]:
station_name = "강동구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_diurnal_profile(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 11.3 강북구

**강북구 × 모든 오염물질의 평균 24시간 패턴**


In [ ]:
station_name = "강북구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_diurnal_profile(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 11.4 강서구

**강서구 × 모든 오염물질의 평균 24시간 패턴**


In [ ]:
station_name = "강서구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_diurnal_profile(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 11.5 관악구

**관악구 × 모든 오염물질의 평균 24시간 패턴**


In [ ]:
station_name = "관악구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_diurnal_profile(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 11.6 광진구

**광진구 × 모든 오염물질의 평균 24시간 패턴**


In [ ]:
station_name = "광진구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_diurnal_profile(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 11.7 구로구

**구로구 × 모든 오염물질의 평균 24시간 패턴**


In [ ]:
station_name = "구로구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_diurnal_profile(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 11.8 금천구

**금천구 × 모든 오염물질의 평균 24시간 패턴**


In [ ]:
station_name = "금천구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_diurnal_profile(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 11.9 노원구

**노원구 × 모든 오염물질의 평균 24시간 패턴**


In [ ]:
station_name = "노원구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_diurnal_profile(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 11.10 도봉구

**도봉구 × 모든 오염물질의 평균 24시간 패턴**


In [ ]:
station_name = "도봉구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_diurnal_profile(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 11.11 동대문구

**동대문구 × 모든 오염물질의 평균 24시간 패턴**


In [ ]:
station_name = "동대문구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_diurnal_profile(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 11.12 동작구

**동작구 × 모든 오염물질의 평균 24시간 패턴**


In [ ]:
station_name = "동작구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_diurnal_profile(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 11.13 마포구

**마포구 × 모든 오염물질의 평균 24시간 패턴**


In [ ]:
station_name = "마포구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_diurnal_profile(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 11.14 서대문구

**서대문구 × 모든 오염물질의 평균 24시간 패턴**


In [ ]:
station_name = "서대문구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_diurnal_profile(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 11.15 서초구

**서초구 × 모든 오염물질의 평균 24시간 패턴**


In [ ]:
station_name = "서초구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_diurnal_profile(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 11.16 성동구

**성동구 × 모든 오염물질의 평균 24시간 패턴**


In [ ]:
station_name = "성동구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_diurnal_profile(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 11.17 성북구

**성북구 × 모든 오염물질의 평균 24시간 패턴**


In [ ]:
station_name = "성북구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_diurnal_profile(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 11.18 송파구

**송파구 × 모든 오염물질의 평균 24시간 패턴**


In [ ]:
station_name = "송파구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_diurnal_profile(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 11.19 양천구

**양천구 × 모든 오염물질의 평균 24시간 패턴**


In [ ]:
station_name = "양천구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_diurnal_profile(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 11.20 영등포구

**영등포구 × 모든 오염물질의 평균 24시간 패턴**


In [ ]:
station_name = "영등포구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_diurnal_profile(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 11.21 용산구

**용산구 × 모든 오염물질의 평균 24시간 패턴**


In [ ]:
station_name = "용산구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_diurnal_profile(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 11.22 은평구

**은평구 × 모든 오염물질의 평균 24시간 패턴**


In [ ]:
station_name = "은평구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_diurnal_profile(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 11.23 종로구

**종로구 × 모든 오염물질의 평균 24시간 패턴**


In [ ]:
station_name = "종로구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_diurnal_profile(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 11.24 중구

**중구 × 모든 오염물질의 평균 24시간 패턴**


In [ ]:
station_name = "중구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_diurnal_profile(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 11.25 중랑구

**중랑구 × 모든 오염물질의 평균 24시간 패턴**


In [ ]:
station_name = "중랑구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_diurnal_profile(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

# 12. 전체 측정소 × 전체 오염물질 평균 1~12월 계절성

2019~2026 전체 기간을 이용해  
측정소별·오염물질별 **전형적인 1~12월 패턴**을 모두 출력한다.


In [ ]:
def plot_month_of_year_profile(
    station,
    pollutant
):

    validate_station_pollutant(
        station,
        pollutant
    )

    temp = monthly[
        monthly["station"]
        == station
    ].copy()

    profile = (
        temp
        .groupby("month")[pollutant]
        .mean()
        .reindex(range(1, 13))
    )

    plt.figure(
        figsize=(10, 5)
    )

    plt.plot(
        profile.index,
        profile.values,
        marker="o"
    )

    plt.title(
        f"{station} | "
        f"{POLLUTANTS[pollutant]} | "
        "평균 월별 계절성"
    )

    plt.xlabel("Month")

    plt.ylabel(
        POLLUTANTS[pollutant]
    )

    plt.xticks(
        range(1, 13)
    )

    plt.grid(
        alpha=0.25
    )

    plt.tight_layout()
    plt.show()

## 12.1 강남구

**강남구 × 모든 오염물질의 평균 1~12월 계절성**


In [ ]:
station_name = "강남구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_month_of_year_profile(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 12.2 강동구

**강동구 × 모든 오염물질의 평균 1~12월 계절성**


In [ ]:
station_name = "강동구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_month_of_year_profile(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 12.3 강북구

**강북구 × 모든 오염물질의 평균 1~12월 계절성**


In [ ]:
station_name = "강북구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_month_of_year_profile(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 12.4 강서구

**강서구 × 모든 오염물질의 평균 1~12월 계절성**


In [ ]:
station_name = "강서구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_month_of_year_profile(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 12.5 관악구

**관악구 × 모든 오염물질의 평균 1~12월 계절성**


In [ ]:
station_name = "관악구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_month_of_year_profile(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 12.6 광진구

**광진구 × 모든 오염물질의 평균 1~12월 계절성**


In [ ]:
station_name = "광진구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_month_of_year_profile(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 12.7 구로구

**구로구 × 모든 오염물질의 평균 1~12월 계절성**


In [ ]:
station_name = "구로구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_month_of_year_profile(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 12.8 금천구

**금천구 × 모든 오염물질의 평균 1~12월 계절성**


In [ ]:
station_name = "금천구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_month_of_year_profile(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 12.9 노원구

**노원구 × 모든 오염물질의 평균 1~12월 계절성**


In [ ]:
station_name = "노원구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_month_of_year_profile(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 12.10 도봉구

**도봉구 × 모든 오염물질의 평균 1~12월 계절성**


In [ ]:
station_name = "도봉구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_month_of_year_profile(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 12.11 동대문구

**동대문구 × 모든 오염물질의 평균 1~12월 계절성**


In [ ]:
station_name = "동대문구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_month_of_year_profile(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 12.12 동작구

**동작구 × 모든 오염물질의 평균 1~12월 계절성**


In [ ]:
station_name = "동작구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_month_of_year_profile(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 12.13 마포구

**마포구 × 모든 오염물질의 평균 1~12월 계절성**


In [ ]:
station_name = "마포구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_month_of_year_profile(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 12.14 서대문구

**서대문구 × 모든 오염물질의 평균 1~12월 계절성**


In [ ]:
station_name = "서대문구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_month_of_year_profile(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 12.15 서초구

**서초구 × 모든 오염물질의 평균 1~12월 계절성**


In [ ]:
station_name = "서초구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_month_of_year_profile(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 12.16 성동구

**성동구 × 모든 오염물질의 평균 1~12월 계절성**


In [ ]:
station_name = "성동구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_month_of_year_profile(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 12.17 성북구

**성북구 × 모든 오염물질의 평균 1~12월 계절성**


In [ ]:
station_name = "성북구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_month_of_year_profile(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 12.18 송파구

**송파구 × 모든 오염물질의 평균 1~12월 계절성**


In [ ]:
station_name = "송파구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_month_of_year_profile(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 12.19 양천구

**양천구 × 모든 오염물질의 평균 1~12월 계절성**


In [ ]:
station_name = "양천구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_month_of_year_profile(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 12.20 영등포구

**영등포구 × 모든 오염물질의 평균 1~12월 계절성**


In [ ]:
station_name = "영등포구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_month_of_year_profile(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 12.21 용산구

**용산구 × 모든 오염물질의 평균 1~12월 계절성**


In [ ]:
station_name = "용산구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_month_of_year_profile(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 12.22 은평구

**은평구 × 모든 오염물질의 평균 1~12월 계절성**


In [ ]:
station_name = "은평구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_month_of_year_profile(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 12.23 종로구

**종로구 × 모든 오염물질의 평균 1~12월 계절성**


In [ ]:
station_name = "종로구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_month_of_year_profile(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 12.24 중구

**중구 × 모든 오염물질의 평균 1~12월 계절성**


In [ ]:
station_name = "중구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_month_of_year_profile(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 12.25 중랑구

**중랑구 × 모든 오염물질의 평균 1~12월 계절성**


In [ ]:
station_name = "중랑구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_month_of_year_profile(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

# 13. 전체 측정소 × 전체 오염물질 연도별 월 패턴

각 측정소·오염물질에서 2019~2026의 각 연도를  
1~12월 축 위에 겹쳐서 그린다.

이를 통해:

- 계절 패턴의 반복성
- 특정 연도의 이상치
- 2026년 현재까지의 위치

를 확인한다.


In [ ]:
def plot_yearly_month_profiles(
    station,
    pollutant
):

    validate_station_pollutant(
        station,
        pollutant
    )

    temp = monthly[
        monthly["station"]
        == station
    ].copy()

    plt.figure(
        figsize=(11, 6)
    )

    for year, part in temp.groupby(
        "year"
    ):

        part = part.sort_values(
            "month"
        )

        plt.plot(
            part["month"],
            part[pollutant],
            marker="o",
            linewidth=1,
            label=str(year)
        )

    plt.title(
        f"{station} | "
        f"{POLLUTANTS[pollutant]} | "
        "연도별 월 패턴"
    )

    plt.xlabel("Month")

    plt.ylabel(
        POLLUTANTS[pollutant]
    )

    plt.xticks(
        range(1, 13)
    )

    plt.grid(
        alpha=0.25
    )

    plt.legend(
        ncol=4
    )

    plt.tight_layout()
    plt.show()

## 13.1 강남구

**강남구 × 모든 오염물질의 연도별 월 패턴**


In [ ]:
station_name = "강남구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_yearly_month_profiles(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 13.2 강동구

**강동구 × 모든 오염물질의 연도별 월 패턴**


In [ ]:
station_name = "강동구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_yearly_month_profiles(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 13.3 강북구

**강북구 × 모든 오염물질의 연도별 월 패턴**


In [ ]:
station_name = "강북구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_yearly_month_profiles(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 13.4 강서구

**강서구 × 모든 오염물질의 연도별 월 패턴**


In [ ]:
station_name = "강서구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_yearly_month_profiles(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 13.5 관악구

**관악구 × 모든 오염물질의 연도별 월 패턴**


In [ ]:
station_name = "관악구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_yearly_month_profiles(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 13.6 광진구

**광진구 × 모든 오염물질의 연도별 월 패턴**


In [ ]:
station_name = "광진구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_yearly_month_profiles(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 13.7 구로구

**구로구 × 모든 오염물질의 연도별 월 패턴**


In [ ]:
station_name = "구로구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_yearly_month_profiles(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 13.8 금천구

**금천구 × 모든 오염물질의 연도별 월 패턴**


In [ ]:
station_name = "금천구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_yearly_month_profiles(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 13.9 노원구

**노원구 × 모든 오염물질의 연도별 월 패턴**


In [ ]:
station_name = "노원구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_yearly_month_profiles(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 13.10 도봉구

**도봉구 × 모든 오염물질의 연도별 월 패턴**


In [ ]:
station_name = "도봉구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_yearly_month_profiles(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 13.11 동대문구

**동대문구 × 모든 오염물질의 연도별 월 패턴**


In [ ]:
station_name = "동대문구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_yearly_month_profiles(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 13.12 동작구

**동작구 × 모든 오염물질의 연도별 월 패턴**


In [ ]:
station_name = "동작구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_yearly_month_profiles(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 13.13 마포구

**마포구 × 모든 오염물질의 연도별 월 패턴**


In [ ]:
station_name = "마포구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_yearly_month_profiles(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 13.14 서대문구

**서대문구 × 모든 오염물질의 연도별 월 패턴**


In [ ]:
station_name = "서대문구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_yearly_month_profiles(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 13.15 서초구

**서초구 × 모든 오염물질의 연도별 월 패턴**


In [ ]:
station_name = "서초구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_yearly_month_profiles(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 13.16 성동구

**성동구 × 모든 오염물질의 연도별 월 패턴**


In [ ]:
station_name = "성동구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_yearly_month_profiles(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 13.17 성북구

**성북구 × 모든 오염물질의 연도별 월 패턴**


In [ ]:
station_name = "성북구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_yearly_month_profiles(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 13.18 송파구

**송파구 × 모든 오염물질의 연도별 월 패턴**


In [ ]:
station_name = "송파구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_yearly_month_profiles(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 13.19 양천구

**양천구 × 모든 오염물질의 연도별 월 패턴**


In [ ]:
station_name = "양천구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_yearly_month_profiles(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 13.20 영등포구

**영등포구 × 모든 오염물질의 연도별 월 패턴**


In [ ]:
station_name = "영등포구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_yearly_month_profiles(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 13.21 용산구

**용산구 × 모든 오염물질의 연도별 월 패턴**


In [ ]:
station_name = "용산구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_yearly_month_profiles(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 13.22 은평구

**은평구 × 모든 오염물질의 연도별 월 패턴**


In [ ]:
station_name = "은평구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_yearly_month_profiles(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 13.23 종로구

**종로구 × 모든 오염물질의 연도별 월 패턴**


In [ ]:
station_name = "종로구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_yearly_month_profiles(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 13.24 중구

**중구 × 모든 오염물질의 연도별 월 패턴**


In [ ]:
station_name = "중구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_yearly_month_profiles(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 13.25 중랑구

**중랑구 × 모든 오염물질의 연도별 월 패턴**


In [ ]:
station_name = "중랑구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_yearly_month_profiles(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

# 14. 전체 측정소 × 전체 오염물질 월별 장기 추세

10번에서도 월별 장기 추세가 포함되지만,  
14번에서는 월별 그래프만 따로 모아 빠르게 비교할 수 있도록 한다.

오염물질은 단위가 다르기 때문에 각각 별도의 figure로 출력한다.


## 14.1 강남구

**강남구 × 모든 오염물질 월별 장기 추세**


In [ ]:
station_name = "강남구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_monthly(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 14.2 강동구

**강동구 × 모든 오염물질 월별 장기 추세**


In [ ]:
station_name = "강동구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_monthly(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 14.3 강북구

**강북구 × 모든 오염물질 월별 장기 추세**


In [ ]:
station_name = "강북구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_monthly(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 14.4 강서구

**강서구 × 모든 오염물질 월별 장기 추세**


In [ ]:
station_name = "강서구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_monthly(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 14.5 관악구

**관악구 × 모든 오염물질 월별 장기 추세**


In [ ]:
station_name = "관악구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_monthly(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 14.6 광진구

**광진구 × 모든 오염물질 월별 장기 추세**


In [ ]:
station_name = "광진구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_monthly(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 14.7 구로구

**구로구 × 모든 오염물질 월별 장기 추세**


In [ ]:
station_name = "구로구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_monthly(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 14.8 금천구

**금천구 × 모든 오염물질 월별 장기 추세**


In [ ]:
station_name = "금천구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_monthly(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 14.9 노원구

**노원구 × 모든 오염물질 월별 장기 추세**


In [ ]:
station_name = "노원구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_monthly(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 14.10 도봉구

**도봉구 × 모든 오염물질 월별 장기 추세**


In [ ]:
station_name = "도봉구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_monthly(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 14.11 동대문구

**동대문구 × 모든 오염물질 월별 장기 추세**


In [ ]:
station_name = "동대문구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_monthly(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 14.12 동작구

**동작구 × 모든 오염물질 월별 장기 추세**


In [ ]:
station_name = "동작구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_monthly(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 14.13 마포구

**마포구 × 모든 오염물질 월별 장기 추세**


In [ ]:
station_name = "마포구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_monthly(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 14.14 서대문구

**서대문구 × 모든 오염물질 월별 장기 추세**


In [ ]:
station_name = "서대문구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_monthly(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 14.15 서초구

**서초구 × 모든 오염물질 월별 장기 추세**


In [ ]:
station_name = "서초구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_monthly(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 14.16 성동구

**성동구 × 모든 오염물질 월별 장기 추세**


In [ ]:
station_name = "성동구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_monthly(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 14.17 성북구

**성북구 × 모든 오염물질 월별 장기 추세**


In [ ]:
station_name = "성북구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_monthly(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 14.18 송파구

**송파구 × 모든 오염물질 월별 장기 추세**


In [ ]:
station_name = "송파구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_monthly(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 14.19 양천구

**양천구 × 모든 오염물질 월별 장기 추세**


In [ ]:
station_name = "양천구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_monthly(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 14.20 영등포구

**영등포구 × 모든 오염물질 월별 장기 추세**


In [ ]:
station_name = "영등포구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_monthly(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 14.21 용산구

**용산구 × 모든 오염물질 월별 장기 추세**


In [ ]:
station_name = "용산구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_monthly(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 14.22 은평구

**은평구 × 모든 오염물질 월별 장기 추세**


In [ ]:
station_name = "은평구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_monthly(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 14.23 종로구

**종로구 × 모든 오염물질 월별 장기 추세**


In [ ]:
station_name = "종로구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_monthly(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 14.24 중구

**중구 × 모든 오염물질 월별 장기 추세**


In [ ]:
station_name = "중구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_monthly(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

## 14.25 중랑구

**중랑구 × 모든 오염물질 월별 장기 추세**


In [ ]:
station_name = "중랑구"

if station_name in stations:

    for pollutant in AVAILABLE_POLLUTANTS:

        print(
            f"{station_name} × "
            f"{POLLUTANTS[pollutant]}"
        )

        plot_monthly(
            station=station_name,
            pollutant=pollutant
        )

else:

    print(
        "⚠ 데이터에 측정소가 없습니다:",
        station_name
    )

# 15. 선택한 측정소끼리 월별 장기 추세 비교

25개 구를 한꺼번에 line chart로 겹치면 읽기 어려우므로 관심 있는 몇 개 측정소만 선택한다.


### 범례

#### 오염물질 코드

| 코드 | 오염물질 | 단위 |
|---|---|---|
| `pm10_1h` | 미세먼지 1시간 | ㎍/㎥ |
| `pm10_24h` | 미세먼지 24시간 | ㎍/㎥ |
| `pm25` | 초미세먼지 | ㎍/㎥ |
| `o3` | 오존 | ppm |
| `no2` | 이산화질소 | ppm |
| `co` | 일산화탄소 | ppm |
| `so2` | 아황산가스 | ppm |

#### 측정소 코드

| 코드 | 측정소 |
|---:|---|
| 0 | 강남구 |
| 1 | 강동구 |
| 2 | 강북구 |
| 3 | 강서구 |
| 4 | 관악구 |
| 5 | 광진구 |
| 6 | 구로구 |
| 7 | 금천구 |
| 8 | 노원구 |
| 9 | 도봉구 |
| 10 | 동대문구 |
| 11 | 동작구 |
| 12 | 마포구 |
| 13 | 서대문구 |
| 14 | 서초구 |
| 15 | 성동구 |
| 16 | 성북구 |
| 17 | 송파구 |
| 18 | 양천구 |
| 19 | 영등포구 |
| 20 | 용산구 |
| 21 | 은평구 |
| 22 | 종로구 |
| 23 | 중구 |
| 24 | 중랑구 |

In [ ]:
COMPARE_STATIONS = [
    "강남구",
    "송파구",
    "서초구"
] # 범례에 맞게 변경

SELECTED_POLLUTANT = "pm25" # 범례에 맞게 변경


def plot_station_comparison_monthly(
    stations_to_compare,
    pollutant
):
    if pollutant not in AVAILABLE_POLLUTANTS:
        raise ValueError(
            f"사용할 수 없는 오염물질입니다: {pollutant}\n"
            f"사용 가능: {AVAILABLE_POLLUTANTS}"
        )

    plt.figure(
        figsize=(14, 6)
    )

    for station in stations_to_compare:

        if station not in stations:
            print(
                f"⚠ 존재하지 않는 측정소: {station}"
            )
            continue

        temp = (
            monthly[
                monthly["station"] == station
            ]
            .sort_values("datetime")
        )

        plt.plot(
            temp["datetime"],
            temp[pollutant],
            linewidth=1,
            label=station
        )

    plt.title(
        f"{POLLUTANTS[pollutant]} | "
        "선택 측정소 월별 추세 비교"
    )

    plt.xlabel("Month")
    plt.ylabel(
        POLLUTANTS[pollutant]
    )

    plt.grid(
        alpha=0.25
    )

    plt.legend()
    plt.tight_layout()
    plt.show()


# ============================================================
# 실행
# ============================================================

plot_station_comparison_monthly(
    COMPARE_STATIONS,
    SELECTED_POLLUTANT
)

# 16. 25개 구 월별 Heatmap

25개 구의 line을 전부 겹치는 대신 heatmap으로 본다.

- 행: 측정소
- 열: 월
- 값: 월평균 농도


### 범례

#### 오염물질 코드

| 코드 | 오염물질 | 단위 |
|---|---|---|
| `pm10_1h` | 미세먼지 1시간 | ㎍/㎥ |
| `pm10_24h` | 미세먼지 24시간 | ㎍/㎥ |
| `pm25` | 초미세먼지 | ㎍/㎥ |
| `o3` | 오존 | ppm |
| `no2` | 이산화질소 | ppm |
| `co` | 일산화탄소 | ppm |
| `so2` | 아황산가스 | ppm |

In [ ]:
def plot_station_month_heatmap(pollutant, start_year=2019, end_year=2026):
    if pollutant not in AVAILABLE_POLLUTANTS:
        raise ValueError(pollutant)

    temp = monthly[monthly['year'].between(start_year, end_year)].copy()
    pivot = (
        temp
        .pivot(index='station', columns='datetime', values=pollutant)
        .sort_index()
    )

    fig, ax = plt.subplots(figsize=(16, max(7, len(pivot) * 0.35)))
    im = ax.imshow(pivot.values, aspect='auto')
    ax.set_title(f'{POLLUTANTS[pollutant]} | {start_year}~{end_year} 측정소 × 월 Heatmap')
    ax.set_ylabel('Station')
    ax.set_yticks(np.arange(len(pivot.index)))
    ax.set_yticklabels(pivot.index)

    step = 6
    tick_pos = np.arange(0, len(pivot.columns), step)
    tick_labels = [pd.Timestamp(pivot.columns[i]).strftime('%Y-%m') for i in tick_pos]
    ax.set_xticks(tick_pos)
    ax.set_xticklabels(tick_labels, rotation=45, ha='right')
    fig.colorbar(im, ax=ax, label=POLLUTANTS[pollutant])
    plt.tight_layout()
    plt.show()

plot_station_month_heatmap('pm10_1h')
plot_station_month_heatmap('pm10_24h')
plot_station_month_heatmap('pm25')
plot_station_month_heatmap('o3')
plot_station_month_heatmap('no2')
plot_station_month_heatmap('co')
plot_station_month_heatmap('so2')

# 17. 서울 평균 대비 지역 편차

같은 월의 서울 25개 측정소 평균을 빼서 지역 자체의 상대적 특성이 지속되는지 본다.

```text
지역 편차 = 해당 측정소 월평균 - 같은 월 서울 전체 측정소 평균
```

- 음수: 같은 달 서울 평균보다 낮음
- 양수: 같은 달 서울 평균보다 높음


### 범례

#### 오염물질 코드

| 코드 | 오염물질 | 단위 |
|---|---|---|
| `pm10_1h` | 미세먼지 1시간 | ㎍/㎥ |
| `pm10_24h` | 미세먼지 24시간 | ㎍/㎥ |
| `pm25` | 초미세먼지 | ㎍/㎥ |
| `o3` | 오존 | ppm |
| `no2` | 이산화질소 | ppm |
| `co` | 일산화탄소 | ppm |
| `so2` | 아황산가스 | ppm |

#### 측정소 코드

| 코드 | 측정소 |
|---:|---|
| 0 | 강남구 |
| 1 | 강동구 |
| 2 | 강북구 |
| 3 | 강서구 |
| 4 | 관악구 |
| 5 | 광진구 |
| 6 | 구로구 |
| 7 | 금천구 |
| 8 | 노원구 |
| 9 | 도봉구 |
| 10 | 동대문구 |
| 11 | 동작구 |
| 12 | 마포구 |
| 13 | 서대문구 |
| 14 | 서초구 |
| 15 | 성동구 |
| 16 | 성북구 |
| 17 | 송파구 |
| 18 | 양천구 |
| 19 | 영등포구 |
| 20 | 용산구 |
| 21 | 은평구 |
| 22 | 종로구 |
| 23 | 중구 |
| 24 | 중랑구 |

In [ ]:
def monthly_relative_to_seoul(pollutant):
    if pollutant not in AVAILABLE_POLLUTANTS:
        raise ValueError(pollutant)

    temp = monthly[['station', 'datetime', 'year', 'month', pollutant]].copy()
    temp['seoul_mean'] = temp.groupby('datetime')[pollutant].transform('mean')
    temp['delta_seoul'] = temp[pollutant] - temp['seoul_mean']
    return temp


def plot_relative_monthly(station, pollutant):
    validate_station_pollutant(station, pollutant)
    temp = monthly_relative_to_seoul(pollutant)
    temp = temp[temp['station'] == station].sort_values('datetime')

    plt.figure(figsize=(14, 5))
    plt.plot(temp['datetime'], temp['delta_seoul'], linewidth=1)
    plt.axhline(0, linewidth=1)
    plt.title(f'{station} | {POLLUTANTS[pollutant]} | 서울 월평균 대비 편차')
    plt.xlabel('Month')
    plt.ylabel('지역 - 서울평균')
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

plot_relative_monthly('강남구', 'pm10_1h')

# 18. 2019~2026 평균적인 지역 편차

In [ ]:
def plot_long_run_station_relative(pollutant):
    temp = monthly_relative_to_seoul(pollutant)
    summary = temp.groupby('station')['delta_seoul'].mean().sort_values()

    plt.figure(figsize=(11, max(7, len(summary) * 0.32)))
    plt.barh(summary.index, summary.values)
    plt.axvline(0, linewidth=1)
    plt.title(f'{POLLUTANTS[pollutant]} | 2019~2026 서울 평균 대비 측정소 평균 편차')
    plt.xlabel('지역 - 서울평균')
    plt.ylabel('Station')
    plt.tight_layout()
    plt.show()

    return summary.to_frame('mean_delta_seoul')

relative_summary = plot_long_run_station_relative('pm10_1h')
display(relative_summary)

# 19. 측정소 × 달(1~12월) 계절성 Heatmap

2019~2026을 평균해 전형적인 1~12월 패턴만 본다.


In [ ]:
def plot_station_seasonality_heatmap(pollutant):
    if pollutant not in AVAILABLE_POLLUTANTS:
        raise ValueError(pollutant)

    seasonality = (
        monthly
        .groupby(['station', 'month'])[pollutant]
        .mean()
        .unstack('month')
        .reindex(columns=range(1, 13))
        .sort_index()
    )

    fig, ax = plt.subplots(figsize=(12, max(7, len(seasonality) * 0.35)))
    im = ax.imshow(seasonality.values, aspect='auto')
    ax.set_title(f'{POLLUTANTS[pollutant]} | 측정소 × 월(1~12) 평균 계절성')
    ax.set_xlabel('Month')
    ax.set_ylabel('Station')
    ax.set_xticks(np.arange(12))
    ax.set_xticklabels(range(1, 13))
    ax.set_yticks(np.arange(len(seasonality.index)))
    ax.set_yticklabels(seasonality.index)
    fig.colorbar(im, ax=ax, label=POLLUTANTS[pollutant])
    plt.tight_layout()
    plt.show()

plot_station_seasonality_heatmap('pm10_1h')

# 20. 분석용 집계 CSV 저장

향후 추천 feature 또는 추가 분석용으로 시간/일/월 집계 결과를 저장한다.


In [ ]:
OUTPUT_DIR = ANALYSIS_DIR

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


hourly.to_csv(
    OUTPUT_DIR
    / "air_hourly_clean_2019_2026.csv",
    index=False,
    encoding="utf-8-sig"
)

daily.to_csv(
    OUTPUT_DIR
    / "air_daily_mean_2019_2026.csv",
    index=False,
    encoding="utf-8-sig"
)

monthly.to_csv(
    OUTPUT_DIR
    / "air_monthly_mean_2019_2026.csv",
    index=False,
    encoding="utf-8-sig"
)

quality_df.to_csv(
    OUTPUT_DIR
    / "air_quality_summary_2019_2026.csv",
    index=False,
    encoding="utf-8-sig"
)

year_station_summary.to_csv(
    OUTPUT_DIR
    / "air_year_station_summary_2019_2026.csv",
    index=False,
    encoding="utf-8-sig"
)


print("저장 완료:")
print(OUTPUT_DIR)

# 21. 분석 해석 순서

## A. 데이터 품질

먼저 확인한다.

1. 각 연도에 25개 측정소가 모두 존재하는가?
2. 특정 연도·측정소에서 결측이 급증하는가?
3. 0값이 특정 기간이나 측정소에 몰려 있는가?
4. 2026년 데이터가 어디까지 존재하는가?

## B. 장기 추세

`월별 장기 추세`를 본다.

- 2019 이후 농도가 장기적으로 올라가는가 / 내려가는가?
- 특정 연도만 비정상적으로 높은가?
- 2026년 현재까지의 수준은 과거와 다른가?

## C. 계절성

`평균 1~12월 계절성`과 `연도별 월 패턴`을 같이 본다.

- 특정 달마다 반복적으로 높아지는가?
- 그 패턴이 해마다 유지되는가?
- 측정소마다 계절성의 크기가 다른가?

## D. 시간대 패턴

`평균 24시간 패턴`을 본다.

- 새벽 / 오전 / 오후 / 저녁 중 반복적인 차이가 존재하는가?
- 오존과 NO₂가 서로 다른 시간 패턴을 보이는가?
- PM 계열에서도 의미 있는 시간 패턴이 존재하는가?

시간별 raw line은 특정 episode를 보는 용도이고, 반복적인 시간 패턴은 평균 24시간 profile로 확인한다.

## E. 지역 차이

다음을 함께 본다.

1. 측정소 × 월 heatmap
2. 서울 평균 대비 지역 편차

- 특정 측정소가 장기간 상대적으로 높은가 / 낮은가?
- 지역 우위가 모든 계절에 유지되는가?
- 특정 달에만 지역 순위가 바뀌는가?

## F. 이전 분석과의 관계

이전 짧은 기간 분석에서는 PM10·PM2.5의 월별 계절성, 오존의 시간대 패턴, 서울 평균 대비 지역 편차 등이 중요한 후보로 보였다.

이번 2019~2026 분석에서는 그 결론을 그대로 사용하지 않고 **가설로만 두고 재검증**한다.

```text
과거 결과
    ↓
가설
    ↓
2019~2026 시각화
    ↓
재검증
```

## 이번 단계의 목표

아직 AirScore를 만드는 것이 아니다.

먼저 다음을 명확하게 파악한다.

```text
지역 차이
시간대 패턴
계절성
장기 추세
데이터 품질
```

그 다음 단계에서 필요한 경우 `gu × month × hour` 형태의 historical air climatology feature table을 만든다.


# 22. 빠른 실행 순서

```text
1~8
→ 데이터 로드 / 정제 / 집계

10
→ 관심 측정소와 오염물질 선택

10.1
→ 최근 한 달 시간별 상세

10.2
→ 2019~2026 일별 흐름

10.3
→ 2019~2026 월별 장기 추세

11
→ 평균 24시간 패턴

12
→ 평균 1~12월 계절성

13
→ 연도별 월 패턴

16
→ 25개 구 전체 월별 heatmap

17~18
→ 서울 평균 대비 지역 편차

19
→ 측정소 × 1~12월 계절성 heatmap
```
